In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from ast import literal_eval

RANDOM_STATE = 42
DATA_DIR = Path('data')

def read_data(filename):
    path = DATA_DIR / filename
    if not path.is_file():
        raise FileNotFoundError(f'Falta {path}. Consulta la preparación de datos en README.md.')
    return pd.read_csv(path, low_memory=False)

In [ ]:
movies = read_data('movies_metadata.csv')
movies.head(4)

<h1>Preprocesamiento de datos</h1>

In [ ]:
#Este código transforma la columna genres, que originalmente podría contener cadenas que representan listas de diccionarios, en una lista real de nombres de géneros.

movies['genres'] = movies['genres'].fillna('[]').apply(literal_eval).apply(lambda x: [i['name'] for i in x] if isinstance(x, list) else [])

In [ ]:
#Selecciono las columnas que me interesan y  muestro el tipo de dato de cada columna. Es útil para tener una visión general de la estructura del datsaet.

movies = movies[['id','title','overview','genres','vote_count','vote_average', 'release_date']]

movies.info()

In [ ]:
#Miro a ver cuantos valores nulos hay
movies.isnull().sum()

In [ ]:
# Limpiar antes de crear el catálogo. Los títulos no son identificadores únicos.
movies['id'] = pd.to_numeric(movies['id'], errors='coerce')
for column in ['vote_count', 'vote_average']:
    movies[column] = pd.to_numeric(movies[column], errors='coerce')
movies = movies.dropna(subset=['id', 'title', 'vote_count', 'vote_average']).copy()
movies['id'] = movies['id'].astype('int64')
movies = movies.drop_duplicates(subset='id', keep='first')
movies['overview'] = movies['overview'].fillna('')
movies.head()

<h1>Sistema de recomendación</h1>

In [ ]:
#Ahora necesitamos una métrica para puntuar o calificar las películas.Como no es lo mismo una película con 3
#votos y una puntuacióin de 9 que una pelicula con 3000 votos y una puntuación de 7, usare la formula de IMDB. Que es la siguiente
#Weighted Rating(WR) = (v ÷ (v+m)) × R + (m ÷ (v+m)) × C . Donde
#v es el número de votos para la película;
#m es el número mínimo de votos requeridos para aparecer en la lista;
#R es la calificación promedio de la película
#C es el voto promedio en todo el informe.

#Ya tengo v  y R. Ahora toca calcular C(voto promedio) 

In [ ]:
C = movies['vote_average'].mean()
C

In [ ]:
# Ahora hay que encontrar un valor para "m", es decir, el número mínimo de votos necesarios para aparecer en la lista. 
#Utilizaré el percentil 95 como  punto de corte. Es decir, para que una película aparezca en las listas, debe tener más votos que al menos el 95% de las películas en la lista.
m= movies['vote_count'].quantile(0.95)
m

In [ ]:
#Ahora creo un nuevo DataFrame movielist que contiene solo las películas que tienen un número de votos mayor o igual a m

movielist = movies.copy().loc[movies['vote_count'] >= m]
movielist.shape

In [ ]:
movielist.head(3)

In [ ]:
def weighted_rating(x, m=m, C=C):
    v, R = x['vote_count'], x['vote_average']
    return (v * R + m * C) / (v + m) if v + m else C

In [ ]:
# Aplico la función weighted_rating a cada fila del DataFrame movielist y almaceno el resultado en una nueva columna llamada 'score'.

movielist['score'] = movielist.apply(weighted_rating, axis=1)

In [ ]:
#Filtro las peliculas segun el valor score de mayor a menor.

movielist = movielist.sort_values('score', ascending=False)

#Print the top 10 movies
movielist[['title', 'vote_count', 'vote_average', 'score']].head(10)

In [ ]:
# Lo muestro de manera grafica según el Score

top_n = 10

plt.figure(figsize=(10, 6))
sns.barplot(x='score', y='title', data=movielist.head(top_n), palette='viridis')
plt.title(f'Top {top_n} películas')
plt.xlabel('Puntuación')
plt.ylabel('Título de la película')
plt.show()


<h1>Recomendaciones por genero</h1>

In [ ]:
#La función get_genre_recommendations toma un género y un número de recomendaciones como entrada. Filtra las películas en el DataFrame movielist que
#pertenecen al género especificado, las ordena por puntuación en orden desdecente y devuelve las películas recomendadas para ese género.
#la función main llama a get_genre_recommendations especificando un género(en este caso, 'Horror') y muestra las recomendaciones. 

def get_genre_recommendations(genre, num_recommendations=10):
    # Filtrar películas por el género especificado
    genre_movies = movielist[movielist['genres'].apply(lambda x: genre in x)]
    
    # Ordenar películas por puntuación
    genre_movies = genre_movies.sort_values('score', ascending=False)
    
    # Mostrar las primeras 'recommendations' películas recomendadas
    recommendations = genre_movies[['title', 'vote_count', 'vote_average', 'score']].head(num_recommendations)
    
    return recommendations


def main():
    # Llama a la función con el género deseado y muestra las recomendaciones
    genre = 'Horror'  # Se puede cambiar al género que se desee como 'Romance', 'Thriller', 'Horror', etc.
    recommendations = get_genre_recommendations(genre, num_recommendations=10)
    
    print(f"Recomendaciones de películas en el género '{genre}':")
    print(recommendations)

if __name__ == "__main__":
    main()

<h1>Recomendacions segun el contenido </h1>

In [ ]:
from sklearn.metrics.pairwise import linear_kernel
from sklearn.feature_extraction.text import TfidfVectorizer

credits = read_data('credits.csv')
credits['id'] = pd.to_numeric(credits['id'], errors='coerce')
credits = credits.dropna(subset=['id']).drop_duplicates(subset='id').copy()
credits['id'] = credits['id'].astype('int64')
credits.head(3)

In [ ]:
print(movielist['id'].dtype)
print(credits['id'].dtype)

In [ ]:
# Verificar si hay valores no numéricos
non_numeric_ids = pd.to_numeric(movielist['id'], errors='coerce').isna()
print(movielist[non_numeric_ids])

In [ ]:
# La deduplicación por ID ya se realizó antes de crear movielist.
assert movielist['id'].is_unique

In [ ]:
movielist.head(3)

In [ ]:
movielist['year'] = pd.to_datetime(movielist['release_date'], errors='coerce').dt.year.astype('Int64')

In [ ]:
movielist.head(3)

In [ ]:
# Convertir la columna 'id' en movies a int64 para futuras operaciones y garantizar la coherencia con credits.
movielist['id'] = movielist['id'].astype('int64')

# Verificar los tipos de datos de 'id' en ambos DataFrames
print(movielist['id'].dtype)
print(credits['id'].dtype)

In [ ]:
merged_df = movielist.merge(credits, on='id', how='left', validate='one_to_one').reset_index(drop=True)

In [ ]:
merged_df.head(3)

In [ ]:
for feature in ['cast', 'crew']:
    merged_df[feature] = merged_df[feature].fillna('[]').apply(literal_eval)

In [ ]:
#Defino la funcion clean_data para procesar y limpiar información y la funcion get_directos para extraer y limpiar el nombre del director.

def clean_data(x):
    if isinstance(x, list):
        cleaned_list = []
        for i in x:
            if isinstance(i, dict) and 'name' in i:
                cleaned_list.append(str.lower(i['name'].replace(" ", "")))
            else:
                print(f"Unexpected value: {i}")  # Imprimir valores inesperados
        return cleaned_list
    elif isinstance(x, str):
        return str.lower(x.replace(" ", ""))
    else:
        return ''

def get_director(x):
    for i in x:
        if i['job'] == 'Director':
            return i['name'].replace(" ", "").lower()  # Extraer el nombre del director que esta bajo la llave 'name'
    return ''

In [ ]:
merged_df['director'] = merged_df['crew'].apply(get_director)
for feature in ['cast', 'director']:
    merged_df[feature] = merged_df[feature].apply(clean_data)
# La sinopsis conserva sus palabras para TF-IDF.
merged_df['overview'] = merged_df['overview'].fillna('').str.lower()

In [ ]:
#Creo una representación combinada de características relevantes para cada película.
def create_soup(x):
    return ' '.join(x['cast']) + ' ' + x['director'] + ' ' + ' '.join(x['genres']) + ' ' + x['overview']

# Crear el 'soup' para cada película
merged_df['soup'] = merged_df.apply(create_soup, axis=1)

In [ ]:
# Utilizar TfidfVectorizer para crear la matriz de características, ignorando las stop words.
tfidf = TfidfVectorizer(stop_words='english') 
tfidf_matrix = tfidf.fit_transform(merged_df['soup'])

# Calcular la matriz de similitud de coseno entre las peliculas.
cosine_sim = linear_kernel(tfidf_matrix, tfidf_matrix)


In [ ]:
merged_df = merged_df.reset_index(drop=True)
indices = pd.Series(merged_df.index, index=merged_df['id'])

def resolve_movie(movie):
    # Un ID distingue remakes y películas con títulos idénticos.
    if isinstance(movie, (int, np.integer)):
        if movie not in indices:
            raise ValueError(f'ID no incluido en el catálogo: {movie}')
        return int(indices.loc[movie])
    matches = merged_df.index[merged_df['title'].str.casefold() == str(movie).strip().casefold()]
    if len(matches) == 0:
        raise ValueError(f'Película no incluida en el catálogo: {movie}')
    if len(matches) > 1:
        options = merged_df.loc[matches, ['id', 'title', 'year']].to_dict('records')
        raise ValueError(f'Título ambiguo; utiliza uno de estos ID: {options}')
    return int(matches[0])

def similar_candidates(movie, limit=25, genre_weight=0.0, similarities=None):
    idx = resolve_movie(movie)
    matrix = cosine_sim if similarities is None else similarities
    target_genres = set(merged_df.loc[idx, 'genres'])
    scores = [(i, float(score) + genre_weight * bool(target_genres.intersection(merged_df.loc[i, 'genres'])))
              for i, score in enumerate(matrix[idx]) if i != idx]
    scores.sort(key=lambda item: (-item[1], item[0]))
    return [i for i, _ in scores[:limit]]

In [ ]:
def get_recommendations(title, cosine_sim=None):
    positions = similar_candidates(title, limit=10, similarities=cosine_sim)
    return merged_df.iloc[positions]['title']

In [ ]:
get_recommendations('The Ring', cosine_sim)

In [ ]:
# Los resultados deben volver a evaluarse tras corregir la tokenización de las sinopsis.

In [ ]:
def improved_recommendations(title, gender_weight=0.1):
    positions = similar_candidates(title, limit=25, genre_weight=gender_weight)
    candidates = merged_df.iloc[positions][['title', 'vote_count', 'vote_average', 'year']].copy()
    if candidates.empty:
        return candidates.assign(wr=pd.Series(dtype=float))
    local_C = candidates['vote_average'].mean()
    local_m = candidates['vote_count'].quantile(0.60)
    qualified = candidates[candidates['vote_count'] >= local_m].copy()
    qualified['wr'] = qualified.apply(lambda row: weighted_rating(row, m=local_m, C=local_C), axis=1)
    return qualified.sort_values('wr', ascending=False).head(10)

In [ ]:
improved_recommendations('The Ring')

In [ ]:
# Comparar estas recomendaciones con una referencia de popularidad antes de extraer conclusiones.

In [ ]:
#Aqui podemos ver la distribución de votos de las películas recomendadas para "The Exorcist".

def plot_recommended_movie_scores(title):
    recommended_movies = improved_recommendations(title)
    plt.figure(figsize=(10, 6))
    sns.histplot(recommended_movies['vote_average'], bins=10, kde=False)
    plt.title(f'Distribucion del Vote Average de recomendaciones de peliculas basadas en "{title}"')
    plt.xlabel('Vote Average')
    plt.ylabel('Count')
    plt.show()

plot_recommended_movie_scores('The Ring')


<h1>Filtro colaborativo</h1>

In [ ]:
from surprise import Reader, Dataset, SVD, BaselineOnly, accuracy
from surprise.model_selection import train_test_split, GridSearchCV, KFold

reader = Reader(rating_scale=(0.5, 5))
ratings = read_data('ratings_small.csv')
ratings.head(3)

In [ ]:
min_rating = ratings.min()
max_rating = ratings.max()
print(min_rating, max_rating)

In [ ]:
# Reservar un test que no participa en la selección de parámetros.
data = Dataset.load_from_df(ratings[['userId', 'movieId', 'rating']], reader)
trainset, testset = train_test_split(data, test_size=0.2, random_state=RANDOM_STATE)
tuning_frame = pd.DataFrame(
    [(trainset.to_raw_uid(u), trainset.to_raw_iid(i), r) for u, i, r in trainset.all_ratings()],
    columns=['userId', 'movieId', 'rating'])
tuning_data = Dataset.load_from_df(tuning_frame, reader)
param_grid = {
    'n_factors': [50, 100, 150, 200],
    'n_epochs': [10, 20, 30],
    'lr_all': [0.002, 0.005, 0.01],
    'reg_all': [0.02, 0.1, 0.2],
    'random_state': [RANDOM_STATE],
}
gs = GridSearchCV(SVD, param_grid, measures=['rmse', 'mae'],
                  cv=KFold(n_splits=3, random_state=RANDOM_STATE), n_jobs=1)
gs.fit(tuning_data)
print('Mejores parámetros:', gs.best_params['rmse'])

In [ ]:
# Evaluar SVD y una referencia sobre el mismo test independiente.
evaluation = {}
for name, model in [('SVD', SVD(**gs.best_params['rmse'])), ('Baseline', BaselineOnly())]:
    model.fit(trainset)
    predictions = model.test(testset)
    evaluation[name] = {'RMSE': accuracy.rmse(predictions, verbose=False),
                        'MAE': accuracy.mae(predictions, verbose=False)}
print(pd.DataFrame(evaluation).T)

# Modelo final para recomendar: entrenado con todos los datos disponibles.
algo = SVD(**gs.best_params['rmse'])
algo.fit(data.build_full_trainset())

In [ ]:
#Calificaciones para el usuario con el ID 1
ratings[ratings['userId'] == 1]

In [ ]:
# Ejemplo de consulta, no una medida de evaluación: el modelo final ya vio estos datos.
algo.predict(1, 2)

In [ ]:
# La evaluación independiente es la de la celda anterior al entrenamiento final.

<h1>Filtro híbrido</h1>

In [ ]:
# Enlazar los identificadores de TMDB con los de MovieLens de forma explícita.

In [ ]:
id_map = read_data('links.csv')[['movieId', 'tmdbId']].rename(columns={'tmdbId': 'id'})
for column in ['movieId', 'id']:
    id_map[column] = pd.to_numeric(id_map[column], errors='coerce')
id_map = id_map.dropna().astype({'movieId': 'int64', 'id': 'int64'})
id_map = id_map.drop_duplicates(subset='id', keep='first')

In [ ]:
indices_map = id_map.set_index('id')['movieId']

In [ ]:
def hybrid(userId, title):
    if userId not in set(ratings['userId']):
        raise ValueError(f'Usuario desconocido: {userId}. Usa get_recommendations para recomendaciones sin historial.')
    seen = set(ratings.loc[ratings['userId'] == userId, 'movieId'])
    positions = similar_candidates(title, limit=len(merged_df))
    # Las posiciones y las películas proceden siempre de la misma tabla.
    candidates = merged_df.iloc[positions][['title', 'vote_count', 'vote_average', 'year', 'id']].copy()
    candidates['movieId'] = candidates['id'].map(indices_map)
    candidates = candidates.dropna(subset=['movieId'])
    candidates['movieId'] = candidates['movieId'].astype('int64')
    candidates = candidates[~candidates['movieId'].isin(seen)].head(25).copy()
    if candidates.empty:
        return candidates.assign(est=pd.Series(dtype=float))
    candidates['est'] = candidates['movieId'].apply(lambda movie_id: algo.predict(userId, int(movie_id)).est)
    return candidates.sort_values('est', ascending=False, kind='stable').head(10)

In [ ]:
hybrid(int(ratings['userId'].iloc[0]), 'Gattaca')

In [ ]:
hybrid(100, 'Gattaca')

In [ ]:
hybrid(112, 'Gattaca')

In [ ]:
# Evaluar también Precision@10 o NDCG@10 con datos reservados antes de afirmar la calidad del híbrido.